In [1]:
import sys
sys.path.append("../")

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Функция для отрисовки результатов
def plot_results(timesteps, train_size, 
                 S_data, I_data, R_data, D_data,
                 S_pred, I_pred, R_pred, D_pred,
                 params):
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('SIRD Model - PINN Prediction vs Real Data', fontsize=16)
    
    # Вертикальная линия разделяющая train/test
    train_test_line = train_size - 0.5
    
    # 1. Susceptible (S)
    ax = axes[0, 0]
    ax.plot(timesteps[:train_size], S_data[:train_size], 'b.', label='Train Data', markersize=3)
    ax.plot(timesteps[train_size:], S_data[train_size:], 'g.', label='Test Data', markersize=3)
    ax.plot(timesteps, S_pred, 'r-', label='PINN Prediction', linewidth=2)
    ax.axvline(x=train_test_line, color='k', linestyle='--', alpha=0.7, label='Train/Test split')
    ax.set_xlabel('Time')
    ax.set_ylabel('Susceptible')
    ax.set_title(f'Susceptible (S)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 2. Infected (I)
    ax = axes[0, 1]
    ax.plot(timesteps[:train_size], I_data[:train_size], 'b.', label='Train Data', markersize=3)
    ax.plot(timesteps[train_size:], I_data[train_size:], 'g.', label='Test Data', markersize=3)
    ax.plot(timesteps, I_pred, 'r-', label='PINN Prediction', linewidth=2)
    ax.axvline(x=train_test_line, color='k', linestyle='--', alpha=0.7, label='Train/Test split')
    ax.set_xlabel('Time')
    ax.set_ylabel('Infected')
    ax.set_title(f'Infected (I)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 3. Recovered (R)
    ax = axes[1, 0]
    ax.plot(timesteps[:train_size], R_data[:train_size], 'b.', label='Train Data', markersize=3)
    ax.plot(timesteps[train_size:], R_data[train_size:], 'g.', label='Test Data', markersize=3)
    ax.plot(timesteps, R_pred, 'r-', label='PINN Prediction', linewidth=2)
    ax.axvline(x=train_test_line, color='k', linestyle='--', alpha=0.7, label='Train/Test split')
    ax.set_xlabel('Time')
    ax.set_ylabel('Recovered')
    ax.set_title(f'Recovered (R)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 4. Dead (D)
    ax = axes[1, 1]
    ax.plot(timesteps[:train_size], D_data[:train_size], 'b.', label='Train Data', markersize=3)
    ax.plot(timesteps[train_size:], D_data[train_size:], 'g.', label='Test Data', markersize=3)
    ax.plot(timesteps, D_pred, 'r-', label='PINN Prediction', linewidth=2)
    ax.axvline(x=train_test_line, color='k', linestyle='--', alpha=0.7, label='Train/Test split')
    ax.set_xlabel('Time')
    ax.set_ylabel('Dead')
    ax.set_title(f'Dead (D)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    # plt.savefig('sird_prediction_results.png', dpi=150)
    plt.show()
    
    return fig

# Функция для отрисовки потерь
def plot_losses(losses):
    plt.figure(figsize=(10, 6))
    plt.plot(losses, 'b-', alpha=0.7, linewidth=1)
    plt.yscale('log')
    plt.xlabel('Epoch')
    plt.ylabel('Loss (log scale)')
    plt.title('Training Loss Over Time')
    plt.grid(True, alpha=0.3)
    # plt.savefig('training_loss.png', dpi=150)
    plt.show()

# Функция для вычисления метрик
def calculate_metrics(S_data, I_data, R_data, D_data, 
                     S_pred, I_pred, R_pred, D_pred, train_size):
    
    # Метрики на тестовой выборке
    test_mask = slice(train_size, None)
    
    mse_S = np.mean((S_data[test_mask] - S_pred[test_mask])**2)
    mse_I = np.mean((I_data[test_mask] - I_pred[test_mask])**2)
    mse_R = np.mean((R_data[test_mask] - R_pred[test_mask])**2)
    mse_D = np.mean((D_data[test_mask] - D_pred[test_mask])**2)
    
    mae_S = np.mean(np.abs(S_data[test_mask] - S_pred[test_mask]))
    mae_I = np.mean(np.abs(I_data[test_mask] - I_pred[test_mask]))
    mae_R = np.mean(np.abs(R_data[test_mask] - R_pred[test_mask]))
    mae_D = np.mean(np.abs(D_data[test_mask] - D_pred[test_mask]))
    
    print("\n" + "="*60)
    print("МЕТРИКИ НА ТЕСТОВОЙ ВЫБОРКЕ:")
    print(f"MSE - S: {mse_S:.2f}, I: {mse_I:.2f}, R: {mse_R:.2f}, D: {mse_D:.2f}")
    print(f"MAE - S: {mae_S:.2f}, I: {mae_I:.2f}, R: {mae_R:.2f}, D: {mae_D:.2f}")
    print("="*60)
    
    return {
        'mse': {'S': mse_S, 'I': mse_I, 'R': mse_R, 'D': mse_D},
        'mae': {'S': mae_S, 'I': mae_I, 'R': mae_R, 'D': mae_D}
    }

# Функция для отрисовки сравнения параметров (если есть истинные значения)
def plot_params_comparison(true_params, learned_params):
    if true_params is None:
        return
    
    params_names = ['β (transmission)', 'γ (recovery)', 'μ (mortality)']
    true_values = [true_params['beta'], true_params['gamma'], true_params['mu']]
    learned_values = [learned_params['beta'], learned_params['gamma'], learned_params['mu']]
    
    x = np.arange(len(params_names))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(10, 6))
    bars1 = ax.bar(x - width/2, true_values, width, label='True', alpha=0.8)
    bars2 = ax.bar(x + width/2, learned_values, width, label='Learned', alpha=0.8)
    
    ax.set_xlabel('Parameters')
    ax.set_ylabel('Values')
    ax.set_title('True vs Learned Parameters')
    ax.set_xticks(x)
    ax.set_xticklabels(params_names)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    # Добавляем значения на столбцы
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax.annotate(f'{height:.3f}',
                       xy=(bar.get_x() + bar.get_width() / 2, height),
                       xytext=(0, 3),
                       textcoords="offset points",
                       ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    # plt.savefig('params_comparison.png', dpi=150)
    plt.show()

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Определяем девайс: cuda если есть, иначе cpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

torch.manual_seed(123)
if device.type == 'cuda':
    torch.cuda.manual_seed(123)
np.random.seed(123)
torch.backends.cudnn.deterministic = True

# Загрузка данных
covid_cases = pd.read_csv('./real_datasets/covid-19_Kouprianov.csv')
# covid_cases = pd.read_csv('./synthetic_datasets/01_baseline_constant.csv')
# covid_cases = pd.read_csv('sird_data_01_006_0003.csv')
# covid_cases = pd.read_csv('simulation.csv')

# Получаем данные
S = covid_cases['S'].values
I = covid_cases['I'].values
D = covid_cases['D'].values
R = covid_cases['R'].values
timesteps = np.arange(len(S)).astype(float)

# Параметры
train_size = 180  # размер обучающей выборки
population = S[0] + I[0] + R[0] + D[0]  # Общая популяция в начале

print(f"Всего точек данных: {len(timesteps)}")
print(f"Обучающих точек: {train_size}")
print(f"Тестовых точек: {len(timesteps) - train_size}")

Using device: cuda
Всего точек данных: 369
Обучающих точек: 180
Тестовых точек: 189


In [7]:
from NEW_PINN.PINN_const import EINN_PINN
            
model1 = EINN_PINN(
                t=timesteps,
                S_data=S,
                I_data=I,
                R_data=R,
                D_data=D,
                population=population,
                train_size=train_size,
                device=device
            )
            
# Обучаем
model1.train_model(n_epoch=10000, lambda_data=0.01, lambda_ode=1.0)
            
# Получаем параметры
final_params = model1.params.get_params_dict()
            
# Предсказание
S_pred1, I_pred1, R_pred1, D_pred1 = model1.predict(timesteps)
            
# Конвертируем в numpy
S_pred = S_pred1.numpy()
I_pred = I_pred1.numpy()
R_pred = R_pred1.numpy()
D_pred = D_pred1.numpy()

Epoch     0 | Loss: 138553248.000000 | Data: 4922422272.000000 | ODE: 89329024.000000
Params: β=0.3000, γ=0.1000, μ=0.0100
---
Epoch  1000 | Loss: 12075750.000000 | Data: 1110575744.000000 | ODE: 969992.625000
Params: β=0.2922, γ=0.1116, μ=0.0107
---
Epoch  2000 | Loss: 8436717.000000 | Data: 739552064.000000 | ODE: 1041196.937500
Params: β=0.2637, γ=0.1407, μ=0.0135
---
Epoch  3000 | Loss: 6608575.500000 | Data: 559466880.000000 | ODE: 1013907.125000
Params: β=0.2290, γ=0.1565, μ=0.0168
---
Epoch  4000 | Loss: 5417821.000000 | Data: 433864896.000000 | ODE: 1079171.875000
Params: β=0.1987, γ=0.1418, μ=0.0197
---
Epoch  5000 | Loss: 4280851.000000 | Data: 318753632.000000 | ODE: 1093315.000000
Params: β=0.1692, γ=0.1195, μ=0.0219
---
Epoch  6000 | Loss: 3269375.000000 | Data: 218261568.000000 | ODE: 1086759.375000
Params: β=0.1431, γ=0.1005, μ=0.0208
---
Epoch  7000 | Loss: 2336916.750000 | Data: 139173760.000000 | ODE: 945179.062500
Params: β=0.1213, γ=0.0859, μ=0.0177
---
Epoch  8000 